# S4. 자세 추정과 동작 인식 (온라인 보충)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/2038prob/defense-ai-textbook-code/blob/main/notebooks/online/%EC%98%A8%EB%9D%BC%EC%9D%B8%EB%B3%B4%EC%B6%A9_S4_%EC%9E%90%EC%84%B8_%EC%B6%94%EC%A0%95%EA%B3%BC_%EB%8F%99%EC%9E%91_%EC%9D%B8%EC%8B%9D.ipynb)

「국방 인공지능」 실습 코드 — 이 장의 파이썬 코드를 순서대로 모았습니다. **맨 위 setup 셀부터 위에서 아래로** 실행하세요(앞 셀에서 정의한 변수를 뒤 셀이 이어 씁니다). ⚠️ 표시가 붙은 셀은 개념 설명용 조각으로, 그대로는 실행되지 않을 수 있습니다.

전체 저장소: https://github.com/2038prob/defense-ai-textbook-code


In [ ]:
# === 실습 환경 준비 (Colab/로컬 공통) — 가장 먼저 실행하세요 ===
# 필요한 패키지를 설치하고(이미 있으면 건너뜀) 공통 유틸을 정의합니다.
import importlib.util, subprocess, sys
for pkg in ["torch", "torchvision", "numpy", "matplotlib", "scipy"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", device)

## MediaPipe Pose Landmarker로 33개 관절 추출하기

### 🔬 실습 — 영상에서 포즈 랜드마크 배열 만들기

아래 코드는 시각화보다 *학습용 배열 생성*에 초점을 둔다. 별도로 준비한 `pose_landmarker.task` 모델 파일 경로를 지정한다.

> ⚠️ **설명용 조각** — 아래는 개념을 보여 주는 코드로, 그대로는 실행되지 않을 수 있습니다. (외부 데이터·모델 파일이 필요합니다)

In [ ]:
import cv2
import mediapipe as mp
import numpy as np

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

options = PoseLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path="pose_landmarker.task"),
    running_mode=RunningMode.VIDEO,
    num_poses=1,
    min_pose_detection_confidence=0.5,
    min_pose_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)

cap = cv2.VideoCapture("input.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
if not fps or fps <= 0:
    fps = 30.0

pose_frames = []
with PoseLandmarker.create_from_options(options) as landmarker:
    frame_id = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB,
                            data=rgb)
        timestamp_ms = int(frame_id * 1000.0 / fps)
        result = landmarker.detect_for_video(mp_image,
                                              timestamp_ms)

        if result.pose_landmarks:
            pose = np.array([
                [lm.x, lm.y, lm.z, lm.visibility]
                for lm in result.pose_landmarks[0]
            ], dtype=np.float32)             # (33,4)
        else:
            pose = np.full((33, 4), np.nan, dtype=np.float32)
        pose_frames.append(pose)
        frame_id += 1

cap.release()
pose_sequence = np.stack(pose_frames)          # (T,33,4)
print(pose_sequence.shape)

## 관절 좌표를 동작 특징으로 바꾸기

### 🔬 실습 — 포즈 정규화와 관절각 계산


In [ ]:
import numpy as np

LEFT_SHOULDER, RIGHT_SHOULDER = 11, 12
LEFT_HIP, RIGHT_HIP = 23, 24


def normalize_pose(pose: np.ndarray) -> np.ndarray:
    """pose: (J,4) = x,y,z,visibility"""
    xy = pose[:, :2].copy()
    hip = np.nanmean(xy[[LEFT_HIP, RIGHT_HIP]], axis=0)
    shoulder = xy[LEFT_SHOULDER] - xy[RIGHT_SHOULDER]
    scale = float(np.linalg.norm(shoulder))
    if not np.isfinite(scale) or scale < 1e-6:
        scale = 1.0
    xy = (xy - hip) / scale
    return np.concatenate([xy, pose[:, 2:4]], axis=1)


def joint_angle(a, b, c) -> float:
    u, v = a - b, c - b
    denom = np.linalg.norm(u) * np.linalg.norm(v)
    if denom < 1e-8:
        return float("nan")
    cosine = np.clip(np.dot(u, v) / denom, -1.0, 1.0)
    return float(np.degrees(np.arccos(cosine)))

# 예: 어깨-팔꿈치-손목의 팔꿈치 각도
pose = np.random.rand(33, 4).astype(np.float32)
pose_n = normalize_pose(pose)
angle = joint_angle(pose_n[11, :2],
                    pose_n[13, :2],
                    pose_n[15, :2])
print(pose_n.shape, round(angle, 1))

## 정적 자세와 시간적 동작을 구분하기

### 🔬 실습 — 프레임 분류 결과에 시간적 디바운싱 적용하기


In [ ]:
from collections import deque
import numpy as np

class TemporalDebouncer:
    def __init__(self, window=12, threshold=0.80, min_hits=8):
        self.history = deque(maxlen=window)
        self.threshold = threshold
        self.min_hits = min_hits

    def update(self, prob_drowsy: float) -> tuple[bool, float]:
        self.history.append(float(prob_drowsy))
        smooth = float(np.mean(self.history))
        hits = sum(p >= self.threshold for p in self.history)
        alarm = (len(self.history) == self.history.maxlen and
                 hits >= self.min_hits and smooth >= self.threshold)
        return alarm, smooth

# 프레임별 모델 출력 예시
monitor = TemporalDebouncer(window=12, threshold=0.80, min_hits=8)
for p in [0.1, 0.2, 0.85, 0.9, 0.4, 0.88, 0.91, 0.93,
          0.87, 0.90, 0.92, 0.94, 0.95, 0.96]:
    alarm, score = monitor.update(p)
    print(f"p={p:.2f}, smooth={score:.2f}, alarm={alarm}")

## 포즈 시퀀스를 이용한 경량 동작 인식

### 🔬 실습 — 관절 좌표 시퀀스를 분류하는 작은 LSTM


In [ ]:
import torch
import torch.nn as nn

class PoseSequenceClassifier(nn.Module):
    def __init__(self, num_joints=33, hidden=64, num_classes=4):
        super().__init__()
        self.rnn = nn.LSTM(input_size=num_joints * 3,
                           hidden_size=hidden,
                           num_layers=2,
                           dropout=0.2,
                           batch_first=True)
        self.head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, num_classes)
        )

    def forward(self, pose_seq):
        # pose_seq: (B, T, J, 3) -> (B, T, 3J)
        b, t, j, c = pose_seq.shape
        x = pose_seq.reshape(b, t, j * c)
        out, _ = self.rnn(x)
        return self.head(out[:, -1])

x = torch.randn(8, 24, 33, 3)  # 8개 클립, 24프레임
model = PoseSequenceClassifier(num_joints=33, num_classes=4)
print(model(x).shape)            # torch.Size([8, 4])

## 평가 지표: PCK, OKS, MPJPE와 시간 안정성

### 🔬 실습 — PCK 계산 함수와 관절별 실패 확인


In [ ]:
import numpy as np


def pck(pred, target, visible, norm_length, alpha=0.2):
    """
    pred, target: (J,2), visible: (J,)
    norm_length: 머리 길이, 몸통 길이 또는 bbox 크기
    """
    pred = np.asarray(pred, dtype=np.float32)
    target = np.asarray(target, dtype=np.float32)
    visible = np.asarray(visible, dtype=bool)
    if norm_length <= 0 or not visible.any():
        raise ValueError("valid norm_length and visible joints required")

    dist = np.linalg.norm(pred - target, axis=1)
    correct = dist <= alpha * norm_length
    score = float(correct[visible].mean())
    return score, dist, correct

J = 5
pred = np.array([[10, 10], [20, 20], [32, 31],
                 [40, 40], [55, 48]], dtype=np.float32)
target = np.array([[10, 11], [22, 20], [30, 30],
                   [50, 40], [50, 50]], dtype=np.float32)
visible = np.array([1, 1, 1, 1, 0], dtype=bool)
score, dist, correct = pck(pred, target, visible,
                           norm_length=40, alpha=0.2)
print("PCK:", score)
print("joint errors:", np.round(dist, 1))

## 실습: 작은 3D CNN으로 입력과 출력 흐름 확인

### 🔬 실습 — Tiny 3D CNN 한 스텝 학습

① **모델 정의**: 시간과 공간을 함께 합성곱한다.


In [ ]:
import torch
import torch.nn as nn

class TinyActionNet(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv3d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm3d(16), nn.ReLU(),
            nn.MaxPool3d((1, 2, 2)),             # 시간은 유지, 공간만 절반
            nn.Conv3d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32), nn.ReLU(),
            nn.MaxPool3d((2, 2, 2)),             # 시간/공간 절반
            nn.AdaptiveAvgPool3d((1, 1, 1))
        )
        self.head = nn.Linear(32, num_classes)

    def forward(self, x):                        # x: (B,C,T,H,W)
        z = self.features(x).flatten(1)
        return self.head(z)

② **난수 클립으로 모양 확인 및 한 번 업데이트**.


In [ ]:
model = TinyActionNet(num_classes=4)
x = torch.randn(4, 3, 8, 64, 64)                 # B=4, T=8
label = torch.tensor([0, 1, 2, 3])

logits = model(x)                                # (4, 4)
loss = nn.CrossEntropyLoss()(logits, label)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
optimizer.zero_grad(); loss.backward(); optimizer.step()
print(logits.shape, round(loss.item(), 3))

---

실습 코드 저장소: https://github.com/2038prob/defense-ai-textbook-code  ·  교재 「국방 인공지능」
